In [ ]:
from matplotlib import pyplot as plt

from spice_segmenter import log_enable

log_enable("INFO")

In [ ]:
from quick_setup import end, start, traj

In [ ]:
from spice_segmenter import config

config.solver_step = 60 * 60

In [13]:

from spice_segmenter import (
    Distance,
    Occultation,
    OccultationTypes,
    PhaseAngle,
    SpiceWindow,
)

# define some properties of the trajectory
jupiter_distance = Distance("JUICE", "JUPITER")
io_distance = Distance("JUICE", "IO")

# define the occultation property
io_occultation = Occultation("JUICE", "JUPITER", "IO")

# phase angle
io_phase_angle = PhaseAngle("JUICE", "IO", "SUN")


# compose the properties into a constraint
c = (
    (io_phase_angle > "5 deg")
    & (io_phase_angle < "90 deg")
    & (jupiter_distance < "700000000.0 km")
    & (io_occultation == OccultationTypes.FULL)
)

# solve the constraint within user defined time window
window = SpiceWindow.from_start_end("2023-06-01", "2031-01-01")

out = c.solve(window)
tab = out.to_pandas()


2025-12-20 18:36:34.860 | INFO     | spice_segmenter.constraint_solver.constraint_solver:get_appropriate_solver:793 - Selected solver <class 'spice_segmenter.constraint_solver.constraint_solver.SpiceEventSolver'> for constraint (Distance of JUPITER from JUICE < 700000000.0 kilometer), of type <class 'spice_segmenter.core.constraints.Constraint'>
2025-12-20 18:36:35.581 | INFO     | spice_segmenter.constraint_solver.constraint_solver:get_appropriate_solver:793 - Selected solver <class 'spice_segmenter.constraint_solver.constraint_solver.SpiceOccultationSolver'> for constraint (Occultation of IO by JUPITER, as seen by JUICE = FULL), of type <class 'spice_segmenter.core.constraints.Constraint'>


In [14]:
print(tab.iloc[::40].to_latex(index=False, sparsify=True))

\begin{tabular}{ll}
\toprule
start & end \\
\midrule
2023-09-01 14:25:47.488000 & 2023-09-01 16:30:52.052000 \\
2023-12-23 18:34:58.143000 & 2023-12-23 20:41:04.321000 \\
2026-12-02 05:05:02.839000 & 2026-12-02 06:11:58.390000 \\
2027-04-01 09:44:42.605000 & 2027-04-01 11:59:34.797000 \\
2027-06-11 03:56:25.949000 & 2027-06-11 06:10:52.439000 \\
2029-04-15 14:27:21.113000 & 2029-04-15 16:32:33.675000 \\
2029-09-21 19:13:47.099000 & 2029-09-21 21:17:20.920000 \\
2029-12-01 13:13:09.202000 & 2029-12-01 15:16:19.580000 \\
2030-02-10 07:12:46.207000 & 2030-02-10 09:15:56.555000 \\
2030-04-22 01:15:35.740000 & 2030-04-22 03:19:10.285000 \\
2030-07-01 19:23:53.315000 & 2030-07-01 21:28:05.280000 \\
2030-09-10 13:38:21.246000 & 2030-09-10 15:43:06.070000 \\
2030-11-20 07:57:41.991000 & 2030-11-20 10:02:43.196000 \\
\bottomrule
\end{tabular}



In [15]:
txt = """# define some properties of the trajectory
jupiter_distance = Distance("JUICE", "JUPITER")
io_distance = Distance("JUICE", "IO")

# define the occultation property
io_occultation = Occultation("JUICE", "JUPITER", "IO")  

# phase angle
io_phase_angle = PhaseAngle("JUICE", "IO", "SUN")"""


txt = """# compose the properties into a constraint
c = (
    (io_phase_angle > "5 deg")
    & (io_phase_angle < "90 deg")
    & (jupiter_distance < "700000000.0 km")
    & (io_occultation == OccultationTypes.FULL)
)"""

txt = """# solve the constraint within user defined time window
window = SpiceWindow.from_start_end("2023-06-01", "2031-01-01")

out = c.solve(window)
out.to_pandas()"""


txt = """
from spice_segmenter import (
    Distance,
    Occultation,
    OccultationTypes,
    PhaseAngle,
    SpiceWindow,
)

# define some properties of the trajectory
jupiter_distance = Distance("JUICE", "JUPITER")
io_distance = Distance("JUICE", "IO")

# define the occultation property
io_occultation = Occultation("JUICE", "JUPITER", "IO")

# phase angle
io_phase_angle = PhaseAngle("JUICE", "IO", "SUN")


# compose the properties into a constraint
c = (
    (io_phase_angle > "5 deg")
    & (io_phase_angle < "90 deg")
    & (jupiter_distance < "700000000.0 km")
    & (io_occultation == OccultationTypes.FULL)
)

# solve the constraint within user defined time window
window = SpiceWindow.from_start_end("2023-06-01", "2031-01-01")

out = c.solve(window)
out.to_pandas()"""


txt = """# Define the time reference event
REF_EVENT = TimeReference("COEV#3", "2025-03-31T00:00:00")

# Populate the timeline
timeline = Timeline(time_reference=REF_EVENT)

# custom procedures can be added to the timeline
timeline += default_power_on("71 h")  # power on 71 hours after the event
timeline += janus_to_config("30 s", goto="ON", open_com=True)  # go to config mode
timeline.add_command(clear_all_jobs, "1 m", comment="Clar all jobs")

# Example of a single filter imaging action
cfg = FastSinglefilterImages(
    prepro=PreproFlags(compress=True, roi=True),
    binning="NO",
    cmp_idx=compression_index_from_ratio(1),
    roi_start_col=0,
    roi_start_row=0,
    roi_size_col=0,
    roi_size_row=0,
    img_cnt=1,
    exp_time=int(round(1.2 * SEC_TO_TICKS)),
)

# The slews triggers the action
slew = ConstSlewImagingAction_1(dtime=30 * SEC_TO_CSEC)

timeline.add_command(cfg, comment="JAN-02 (RADIOMETRIC MULTI FILTER)")
timeline.add_command(slew)

# Store job
timeline.add_command(StoreJob(1), comment="Store JOB 1")

# Execute job, anc close com
timeline.add_command(StartSci(1), "1 m", comment="Exec images during slew")
timeline.add_command(ComClose(n_csteps=58), "1 m", comment="Close COM")

# Power off JANUS
timeline += janus_off("10 m")

# Export to compatible formats
timeline.to_pandas().to_excel("Example1.xlsx")  # for the user
timeline.expand().to_file("Example1.tcl")  # to be run on REM
timeline.to_file("Example1.csv")  # For SPOT injestion

timeline.to_pandas()  # as a pandas table"""

txt = """# LJC34180 - JANCoverArm - Arm for cover movement
::SEQM::waittimex 10

::TCML::tcsendx LJC34180 

# LJC34181 - JANCoverMv - Open or Close the cover
::SEQM::waittimex 1

::TCML::tcsendx LJC34181 [list LJP34182 OPEN] \
                          [list LJP34183 42]

# LJC35061 - JANClearImgJob - Clear image job current or all
# Clar all jobs
::SEQM::waittimex 60

::TCML::tcsendx LJC35061 [list LJP35003 ALL]

# LJC35077 - JANSingle - Single-filter image with repetition
# JAN-02 (RADIOMETRIC MULTI FILTER)
::SEQM::waittimex 1

::TCML::tcsendx LJC35077 [list LJP35002 48] \
                          [list LJP14801 NO] \
                          [list LJP14802 0] \
                          [list LJP14803 0] \
                          [list LJP14804 0] \
                          [list LJP14805 0] \
                          [list LJP35008 5418] \
                          [list LJP35011 0] \
                          [list LJP35010 1] \
                          [list LJP35028 1] \
                          [list LJP35029 0]

# LJC35071 - JANSlew - Multi filter slew with fixed parameter
::SEQM::waittimex 1

::TCML::tcsendx LJC35071 [list LJP35020 1] \
                          [list LJP35021 3000]

# LJC35060 - JANStoreImgJob - Store image job at position parameter
# Store JOB 1
::SEQM::waittimex 1

::TCML::tcsendx LJC35060 [list LJP35001 1]"""

In [16]:
from pygments.formatters import LatexFormatter
from pygments.lexers import PythonLexer, TclLexer
from pygments.styles import get_style_by_name

style = get_style_by_name("manni")

lex = PythonLexer()

lex = TclLexer()

fmt = LatexFormatter(style=style)

print(fmt.get_style_defs())


\makeatletter
\def\PY@reset{\let\PY@it=\relax \let\PY@bf=\relax%
    \let\PY@ul=\relax \let\PY@tc=\relax%
    \let\PY@bc=\relax \let\PY@ff=\relax}
\def\PY@tok#1{\csname PY@tok@#1\endcsname}
\def\PY@toks#1+{\ifx\relax#1\empty\else%
    \PY@tok{#1}\expandafter\PY@toks\fi}
\def\PY@do#1{\PY@bc{\PY@tc{\PY@ul{%
    \PY@it{\PY@bf{\PY@ff{#1}}}}}}}
\def\PY#1#2{\PY@reset\PY@toks#1+\relax+\PY@do{#2}}

\@namedef{PY@tok@w}{\def\PY@tc##1{\textcolor[rgb]{0.73,0.73,0.73}{##1}}}
\@namedef{PY@tok@c}{\let\PY@it=\textit\def\PY@tc##1{\textcolor[rgb]{0.00,0.60,1.00}{##1}}}
\@namedef{PY@tok@cp}{\def\PY@tc##1{\textcolor[rgb]{0.00,0.60,0.60}{##1}}}
\@namedef{PY@tok@cs}{\let\PY@bf=\textbf\let\PY@it=\textit\def\PY@tc##1{\textcolor[rgb]{0.00,0.60,1.00}{##1}}}
\@namedef{PY@tok@k}{\let\PY@bf=\textbf\def\PY@tc##1{\textcolor[rgb]{0.00,0.40,0.60}{##1}}}
\@namedef{PY@tok@kp}{\def\PY@tc##1{\textcolor[rgb]{0.00,0.40,0.60}{##1}}}
\@namedef{PY@tok@kt}{\let\PY@bf=\textbf\def\PY@tc##1{\textcolor[rgb]{0.00,0.47,0.53}{##1}}}


In [17]:
import pygments

print(pygments.highlight(txt, formatter=fmt, lexer=lex))

\begin{Verbatim}[commandchars=\\\{\}]
\PY{c}{\PYZsh{}}\PY{c}{ LJC34180 \PYZhy{} JANCoverArm \PYZhy{} Arm for cover movement}
\PY{o}{:}\PY{o}{:}\PY{n+nv}{SEQM}\PY{o}{:}\PY{o}{:}waittimex\PY{+w}{ }\PY{l+m+mi}{10}

\PY{o}{:}\PY{o}{:}\PY{n+nv}{TCML}\PY{o}{:}\PY{o}{:}tcsendx\PY{+w}{ }LJC34180\PY{+w}{ }

\PY{err}{\PYZsh{}}\PY{+w}{ }LJC34181\PY{+w}{ }\PY{o}{\PYZhy{}}\PY{+w}{ }JANCoverMv\PY{+w}{ }\PY{o}{\PYZhy{}}\PY{+w}{ }Open\PY{+w}{ }or\PY{+w}{ }Close\PY{+w}{ }the\PY{+w}{ }cover
\PY{o}{:}\PY{o}{:}\PY{n+nv}{SEQM}\PY{o}{:}\PY{o}{:}waittimex\PY{+w}{ }\PY{l+m+mi}{1}

\PY{o}{:}\PY{o}{:}\PY{n+nv}{TCML}\PY{o}{:}\PY{o}{:}tcsendx\PY{+w}{ }LJC34181\PY{+w}{ }\PY{k}{[}\PY{n+nb}{list}\PY{+w}{ }LJP34182\PY{+w}{ }OPEN\PY{k}{]}\PY{+w}{                           }\PY{k}{[}\PY{n+nb}{list}\PY{+w}{ }LJP34183\PY{+w}{ }\PY{l+m+mi}{42}\PY{k}{]}

\PY{c}{\PYZsh{}}\PY{c}{ LJC35061 \PYZhy{} JANClearImgJob \PYZhy{} Clear image job current or all}
\PY{c}{\PYZsh{}}\PY{c}{ Clar all jobs}
\PY{o}{:}\PY{o}{:}\PY{n+nv}{SEQM}\

In [18]:
status = c(traj.ets) # compute the status of the constraint over the trajectory

fig, ax = plt.subplots(figsize=(18,4))
ax.plot(traj.utc, status, label="OCC status")

ax.set_ylim(-1, 3)

ax1 = plt.gca()


ax2 = ax1.twinx()

a = ax2.plot(traj.utc, traj.dist, label="distance", color="red")

ax3 = ax1.twinx()
ax3.spines.right.set_position(("axes", 1.03))

b = ax3.plot(traj.utc, traj.phase, label="phase angle", color="green")

plt.legend(a+b, [l.get_label() for l in a+b])

2025-12-20 18:36:37.506 | WARNING  | spice_segmenter.core.constraints:__call__:300 - Comparing radian with degree. This is not recommended. Will attempt automatic conversion.
2025-12-20 18:36:37.791 | WARNING  | spice_segmenter.core.constraints:__call__:300 - Comparing radian with degree. This is not recommended. Will attempt automatic conversion.


SpiceNOFRAMECONNECT: 
================================================================================

Toolkit version: CSPICE_N0067

SPICE(NOFRAMECONNECT) --

At epoch 7.3474820684300E+08 TDB (2023 APR 14 12:43:26.843 TDB), there is insufficient information available to transform from reference frame 1 (J2000) to reference frame -28000 (JUICE_SPACECRAFT).

spkpos_c --> SPKPOS --> SPKEZP --> SPKGPS --> REFCHG

================================================================================

In [ ]:
# render the tree of constraints
c.render_tree()

In [ ]:
status = c(traj.ets) # compute the status of the constraint over the trajectory

fig, ax = plt.subplots(figsize=(18,4))
ax.plot(traj.utc, status, label="OCC status")

ax.set_ylim(-1, 3)

ax1 = plt.gca()


ax2 = ax1.twinx()

a = ax2.plot(traj.utc, traj.dist, label="distance", color="red")

ax3 = ax1.twinx()
ax3.spines.right.set_position(("axes", 1.03))

b = ax3.plot(traj.utc, traj.phase, label="phase angle", color="green")

plt.legend(a+b, [l.get_label() for l in a+b])


In [ ]:
# if we want to get the exact intervals, better use a solver and find the precise start and end

import numpy as np

from spice_segmenter.spice_window import SpiceWindow

shift = np.timedelta64(1, "D")
w = SpiceWindow() # a window from start to end of the trajectory


w.add_interval(start+shift, end-shift) # add the interval from start to end of the trajectory

w

In [ ]:
result = c.solve(w)
result

In [ ]:
status = c(traj.ets)

fig, ax = plt.subplots(figsize=(18,4))

result.plot(ax=ax, color="red", label="constraint", alpha=0.3) # plot the results

ax.plot(traj.utc, status)

ax.set_ylim(-1, 3)

ax1 = plt.gca()


ax2 = ax1.twinx()

a = ax2.plot(traj.utc, traj.dist, label="distance", color="red")

ax3 = ax1.twinx()
ax3.spines.right.set_position(("axes", 1.03))

b = ax3.plot(traj.utc, traj.phase, label="phase angle", color="green")

plt.legend(a+b, [l.get_label() for l in a+b])

In [ ]:
result.to_datetimerange() # as datetimerange

In [ ]:
result.to_pandas() # as pandas dataframe

In [ ]:
result.to_juice_core_csv("result.csv") # as juice core csv